# 02 — Profile culture-organism grain

## Goal

The next grain is one source culture-organism combination. A positive culture can have more than one organism label, so this is more detailed than `fct_culture`.

ARMD does not provide a verified isolate identifier. I keep the source culture-organism grain as-is and treat repeated antibiotic records as a data-quality issue to investigate rather than assuming they represent separate isolates.

In [1]:
%sql
select
    count(*) as culture_organism_rows,
    count(distinct culture_organism_key) as distinct_keys,
    count(distinct culture_key) as cultures_with_organisms,
    sum(
        case
            when has_repeated_antibiotic_records then 1
            else 0
        end
    ) as groups_with_repeated_antibiotic_records
from workspace.micro_dev.fct_culture_organism;

,culture_organism_rows,distinct_keys,cultures_with_organisms,groups_with_repeated_antibiotic_records
0,133246,133246,118767,4546


### Grain check

`fct_culture_organism` has 133,246 rows and 133,246 distinct keys. It covers 118,767 culture orders with identified organisms.

There are 4,546 culture-organism groups with repeated antibiotic records. I carry that forward as a reconciliation problem instead of trying to collapse it here.

In [2]:
%sql
select
    organism,
    count(*) as culture_count
from workspace.micro_dev.fct_culture_organism
group by organism
order by culture_count desc
limit 50;

,organism,culture_count
0,ESCHERICHIA COLI,61348
1,KLEBSIELLA PNEUMONIAE,10369
2,ENTEROCOCCUS SPECIES,9462
3,STAPHYLOCOCCUS AUREUS,8415
4,PSEUDOMONAS AERUGINOSA,5745
5,PROTEUS MIRABILIS,4750
6,MUCOID PSEUDOMONAS AERUGINOSA,3324
7,STREPTOCOCCUS AGALACTIAE (GROUP B),2743
8,COAG NEGATIVE STAPHYLOCOCCUS,2230
9,KLEBSIELLA OXYTOCA,1671


### Common source labels

This is the vocabulary exactly as it appears in ARMD. At this point I am not standardizing names; I just want to see which labels drive most of the data and which ones are likely to matter for the reporting layer.

In [3]:
%sql
select
    count(distinct organism) as distinct_source_organism_names
from workspace.micro_dev.fct_culture_organism;

,distinct_source_organism_names
0,314


### Vocabulary size

There are 314 distinct organism labels in this release. That includes species names, complexes, groups, historical names, phenotype-qualified labels, and morphotypes.

In [4]:
%sql
select
    organism,
    count(*) as culture_count
from workspace.micro_dev.fct_culture_organism
group by organism
order by organism;

,organism,culture_count
0,ABIOTROPHIA SPECIES (OR GRANULICATELLA SPECIES),6
1,ACHROMOBACTER SPECIES,31
2,ACHROMOBACTER XYLOSOXIDANS,489
3,ACID FAST BACILLI,2
4,ACID FAST BACILLI DETECTED IN LIQUID MEDIUM,3
5,ACINETOBACTER BAUMANNII,48
6,ACINETOBACTER BAUMANNII COMPLEX,96
7,ACINETOBACTER HAEMOLYTICUS,2
8,ACINETOBACTER LWOFFI,42
9,ACINETOBACTER SPECIES,152


### Takeaway

The culture-organism grain is unique, but the vocabulary is not ready to use directly for reporting. I need an explicit mapping layer so that terminology decisions are version-controlled instead of buried in ad hoc string cleanup.